# Task 11 — Open-Weight LLM Reference (`Qwen/Qwen2.5-7B-Instruct`)

**Protocol:** `phase2-v1` | **Split pool:** `train 10%` (`seed 0`) | **Decoding:** `temperature=0.0` (greedy)

Quy trình thực thi tuân thủ đúng giao thức:
1. **Setup & GPU check:** tự chọn `bf16` (A100/L4) hoặc `4bit` (nếu GPU < 24GB VRAM, cần `bitsandbytes`), ghi rõ precision.
2. **Verify model version & license:** xác minh revision SHA và giấy phép trên Hugging Face trước khi chạy.
3. **Smoke test (`dev-only`, 20 câu):** kiểm tra prompt, sinh nhãn greedy và parser.
4. **Dev sweep (`dev-only`):** thử `zero_shot` + **3 chiến lược few-shot** (`stratified_fixed`, `tfidf_knn`, `balanced_tfidf_knn`) × 2 kiểu prompt (`concise`, `rubric`) lấy ví dụ từ `train 10%`; khóa `selection_llm_{domain}.json`.
5. **Test evaluation (1 lần duy nhất):** chạy cấu hình `zero_shot` và `few_shot` đã khóa trên `test` (`vsmec` 692 câu, `neuesc` 6.613 câu).
6. **Cost benchmark:** đo `total/trainable params`, peak VRAM, latency (`batch=1`), throughput (`batch=16`/`32`) và `GPU-hours / 1000 samples`.

> ⚠️ Bật **Runtime → Change runtime type → GPU** trước khi chạy.

## 0. Setup & GPU Detection

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/PhuongNgoVan/Xdomain-Transfer-Learning.git'
BRANCH = 'task11-xlmr-llm'
WORK_DIR = '/content/Xdomain-Transfer-Learning'
OUT_DIR = 'results/phase2/llm'
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

if not os.path.isdir(WORK_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    '--no-checkout', REPO_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate>=0.34.0', 'huggingface_hub>=0.25.0'], check=True)
print('Setup complete.')

In [ ]:
import subprocess, sys, torch

assert torch.cuda.is_available(), 'Enable GPU runtime in Colab.'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_name = torch.cuda.get_device_name(0)

if vram_gb >= 22.0 and torch.cuda.is_bf16_supported():
    PRECISION = 'bf16'
    BATCH_SIZE = 32
else:
    PRECISION = '4bit'
    BATCH_SIZE = 8
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.43.0'], check=True)

print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | Selected precision: {PRECISION} | batch_size: {BATCH_SIZE}')

## 1. Verify Model Version & License + Load Model

In [ ]:
import json
from src.phase2.llm import HuggingFaceLLMClassifier, verify_hf_model_metadata

hub_info = verify_hf_model_metadata(MODEL_NAME, verify_license=True)
print('Verified HF Hub metadata:', json.dumps(hub_info, indent=2))

clf = HuggingFaceLLMClassifier(
    model_name=MODEL_NAME,
    precision=PRECISION,
    max_input_length=1024,
    max_new_tokens=16,
    batch_size=BATCH_SIZE,
    verify_license=True,
)
print('Loaded model metadata:', json.dumps(clf.metadata(), indent=2))

## 2. Smoke Test (20 dev samples, `dev-only`)

In [ ]:
from src.phase2.llm import run_llm_dev_sweep

smoke_vsmec = run_llm_dev_sweep(
    'vsmec',
    clf.generate_batch,
    tokenizer=clf.tokenizer,
    model_meta=clf.metadata(),
    strategies=('zero_shot', 'balanced_tfidf_knn'),
    prompt_styles=('concise',),
    shots_per_class=1,
    out_dir='results/phase2/llm_smoke',
    max_examples=20,
)
print(smoke_vsmec['sweep_df'][['strategy', 'prompt_style', 'dev_selection_score', 'invalid_output_rate']])

## 3. Dev Sweep — Prompt & Few-Shot Strategy Selection (`dev-only`)

In [ ]:
from src.phase2.llm import load_llm_task_config, run_llm_dev_sweep

cfg = load_llm_task_config('configs/task11_llm.yaml')['llm']

for domain in ('vsmec', 'neuesc'):
    shots = int(cfg['shots_per_class'][domain])
    print(f'\n=== Dev Sweep: {domain} (shots_per_class={shots}) ===')
    res = run_llm_dev_sweep(
        domain,
        clf.generate_batch,
        tokenizer=clf.tokenizer,
        model_meta=clf.metadata(),
        strategies=tuple(cfg['strategies']),
        prompt_styles=tuple(cfg['prompt_styles']),
        shots_per_class=shots,
        seed=0,
        out_dir=OUT_DIR,
    )
    print(res['sweep_df'][['strategy', 'prompt_style', 'shots', 'dev_selection_score', 'invalid_output_rate', 'elapsed_seconds']])
    print('Locked selection:', res['selection']['selected'])

## 4. Final Test Evaluation (Locked Zero-Shot & Few-Shot)

In [ ]:
from pathlib import Path
from src.phase2.llm import run_llm_test_eval

for domain in ('vsmec', 'neuesc'):
    sel_path = Path(OUT_DIR) / domain / f'selection_llm_{domain}.json'
    print(f'\n=== Test Evaluation: {domain} ===')
    out = run_llm_test_eval(
        domain,
        sel_path,
        clf.generate_batch,
        tokenizer=clf.tokenizer,
        model_meta=clf.metadata(),
        out_dir=OUT_DIR,
    )
    print(out['summary_df'][['mode', 'strategy', 'prompt_style', 'label_view', 'map_method', 'macro_f1', 'weighted_f1', 'invalid_output_rate']])

## 5. Standardized GPU Cost Benchmark

In [ ]:
import json
from pathlib import Path
from src.phase2.llm import benchmark_llm_inference, load_llm_splits

for domain in ('vsmec', 'neuesc'):
    texts = load_llm_splits(domain, include_test=False)['dev_texts']
    cost = benchmark_llm_inference(
        clf,
        texts,
        domain,
        n_b1=50,
        large_batch_size=BATCH_SIZE,
        n_large=min(160, len(texts)),
        out_path=Path(OUT_DIR) / f'cost_llm_{domain}.json',
    )
    print(f'=== Cost ({domain}) ===')
    print(json.dumps(cost, indent=2, ensure_ascii=False))

## 6. Archive & Download Results

In [ ]:
!cd {WORK_DIR} && tar czf /content/task11_llm_results.tar.gz results/phase2/llm/

from google.colab import files
files.download('/content/task11_llm_results.tar.gz')